<a href="https://colab.research.google.com/github/kuz-hub/crm-cw1-sp500/blob/main/CW1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install yfinance

In [3]:
import pandas as pd
import yfinance as yf
import datetime
import os
import requests
from io import StringIO
os.makedirs("data/raw", exist_ok=True)
html = requests.get("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies", headers={"User-Agent": "Mozilla/5.0"}).text
tables = pd.read_html(StringIO(html))
tickers_df = tables[0]
tickers_df["Symbol"] = tickers_df["Symbol"].str.replace(".", "-", regex=False)
retrieval_date = datetime.date.today()
print("Retrieval date:", retrieval_date)
tickers_df.to_csv(f"data/raw/tickers_{retrieval_date}.csv", index=False)
print(tickers_df.shape)
tickers_df.head()

Retrieval date: 2026-11-08
(503, 8)


,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989


In [5]:
tickers = tickers_df["Symbol"].tolist()
all_symbols = tickers + ["^GSPC"]
data = yf.download(all_symbols, start="2021-01-01", end="2025-01-01", auto_adjust=False)

[*********************100%***********************]  504 of 504 completed
ERROR:yfinance:
5 Failed downloads:
ERROR:yfinance:['FDXF', 'Q', 'VYLR', 'HONA', 'SNDK']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


In [6]:
adj_close = data["Adj Close"]
close = data["Close"]
volume = data["Volume"]
print(adj_close.shape, close.shape, volume.shape)
print(adj_close.index.min(), adj_close.index.max())
print(adj_close.columns[adj_close.isna().all()].tolist())
print("yfinance", yf.__version__, "pandas", pd.__version__)

(1005, 504) (1005, 504) (1005, 504)
2021-01-04 00:00:00 2024-12-31 00:00:00
['FDXF', 'HONA', 'Q', 'SNDK', 'VYLR']
yfinance 0.2.66 pandas 2.2.3


In [7]:
adj_close.to_csv("data/raw/adj_close_daily.csv")
close.to_csv("data/raw/close_daily.csv")
volume.to_csv("data/raw/volume_daily.csv")
print(os.listdir("data/raw"))

['volume_daily.csv', 'close_daily.csv', 'adj_close_daily.csv', 'tickers_2026-11-08.csv']
